## Configuración Google Colab
> **Solo si estás en Colab:** ejecuta la siguiente celda para montar Google Drive.
> Si trabajas en local (VS Code / Jupyter), puedes saltártela.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    # Replace 'archive' with the name of the folder you uploaded to Drive
    RUTA_DRIVE = '/content/drive/MyDrive/archive'
    os.chdir(RUTA_DRIVE)
    print(f"Directorio: {os.getcwd()}")
    # Install dependencies

    import subprocess
    subprocess.run(['pip', 'install', '-q', 'seaborn', 'scipy'])
else:
    import os
    print(f"Entorno local — directorio: {os.getcwd()}")

# EDA Proyecto I — Fase 3: Análisis Estadístico y Detección de Sesgos
**DataTalent Solutions S.L.** | Módulo II: Análisis y Visualización de Datos

En esta fase calculamos estadísticos descriptivos, exploramos correlaciones, analizamos grupos con `.groupby()` y — lo más crítico — **identificamos sesgos presentes en el dataset** con reflexión sobre su impacto en decisiones de negocio y modelos predictivos.

## 1. Carga de datos limpios

In [43]:
import pandas as pd
import numpy as np
from scipy import stats
import re
import ast
import warnings
warnings.filterwarnings('ignore')

# Load clean data
df     = pd.read_csv('data_roles_completo.csv', low_memory=False)
df_sal = pd.read_csv('data_roles_salario.csv',  low_memory=False)

print(f"Dataset completo (roles de datos): {df.shape}")
print(f"Dataset con salario limpio:        {df_sal.shape}")

Dataset completo (roles de datos): (2916, 39)
Dataset con salario limpio:        (882, 39)


## 2. Estadística descriptiva — Variables clave

### 2.1 Salario anual (salary_annual)

In [44]:
s = df_sal['salary_annual']

print("=== Salario Anual (USD) ===")
print(f"  Recuento           : {s.count():>12,}")
print(f"  Media              : ${s.mean():>12,.0f}")
print(f"  Mediana (P50)      : ${s.median():>12,.0f}")
print(f"  Desviación estándar: ${s.std():>12,.0f}")
print(f"  Mínimo             : ${s.min():>12,.0f}")
print(f"  Percentil 25 (Q1)  : ${s.quantile(0.25):>12,.0f}")
print(f"  Percentil 75 (Q3)  : ${s.quantile(0.75):>12,.0f}")
print(f"  Máximo             : ${s.max():>12,.0f}")
print(f"  Asimetría (skew)   :  {s.skew():>12.3f}")
print(f"  Curtosis           :  {s.kurtosis():>12.3f}")

=== Salario Anual (USD) ===
  Recuento           :          882
  Media              : $     150,096
  Mediana (P50)      : $     141,120
  Desviación estándar: $      53,688
  Mínimo             : $      35,360
  Percentil 25 (Q1)  : $     110,240
  Percentil 75 (Q3)  : $     184,400
  Máximo             : $     302,000
  Asimetría (skew)   :         0.582
  Curtosis           :        -0.190


**Interpretación:** La diferencia positiva entre media y mediana confirma una **distribución con cola derecha**: existen algunos salarios muy altos que elevan la media por encima del valor típico. La asimetría positiva (skew > 0) lo cuantifica. Para orientar candidatos, la **mediana es el indicador más representativo** del salario que esperaría una persona con ese rol, ya que no está distorsionado por los valores extremos.

### 2.2 Vistas de oferta (views)

In [45]:
v = df['views']
print("=== Vistas por Oferta ===")
print(f"  Media              : {v.mean():>10,.1f}")
print(f"  Mediana (P50)      : {v.median():>10,.1f}")
print(f"  Desviación estándar: {v.std():>10,.1f}")
print(f"  P25                : {v.quantile(0.25):>10,.1f}")
print(f"  P75                : {v.quantile(0.75):>10,.1f}")
print(f"  Máximo             : {v.max():>10,.1f}")

=== Vistas por Oferta ===
  Media              :       40.7
  Mediana (P50)      :        8.0
  Desviación estándar:       94.0
  P25                :        4.0
  P75                :       34.0
  Máximo             :    1,393.0


**Interpretación:** La alta desviación estándar respecto a la media indica gran variabilidad: algunas ofertas reciben miles de vistas mientras la mayoría pasa desapercibida. Esto refleja el **efecto de reputación de marca**: las grandes empresas tech concentran la atención.

### 2.3 Solicitudes recibidas (applies)

In [46]:
a = df['applies']
print("=== Solicitudes por Oferta ===")
print(f"  Media              : {a.mean():>10,.1f}")
print(f"  Mediana (P50)      : {a.median():>10,.1f}")
print(f"  Desviación estándar: {a.std():>10,.1f}")
print(f"  P25                : {a.quantile(0.25):>10,.1f}")
print(f"  P75                : {a.quantile(0.75):>10,.1f}")
print(f"  Máximo             : {a.max():>10,.1f}")

=== Solicitudes por Oferta ===
  Media              :       15.1
  Mediana (P50)      :        7.0
  Desviación estándar:       33.4
  P25                :        7.0
  P75                :        7.0
  Máximo             :      566.0


**Interpretación:** La mediana de solicitudes es considerablemente más baja que la media, lo que confirma que pocas ofertas concentran la mayoría de las candidaturas (distribución de Pareto). Esto tiene implicaciones directas para DataTalent: los candidatos deben diferenciarse para destacar en las ofertas con mayor competencia.

## 3. Matriz de correlaciones (.corr())

In [47]:
num_cols = [c for c in ['salary_annual','views','applies','employee_count','follower_count']
            if c in df_sal.columns]

corr = df_sal[num_cols].corr()
print("Matriz de correlaciones (Pearson):")
print(corr.round(3).to_string())

Matriz de correlaciones (Pearson):
                salary_annual  views  applies  employee_count  follower_count
salary_annual           1.000 -0.159   -0.140           0.223           0.244
views                  -0.159  1.000    0.903          -0.046          -0.013
applies                -0.140  0.903    1.000          -0.066          -0.039
employee_count          0.223 -0.046   -0.066           1.000           0.871
follower_count          0.244 -0.013   -0.039           0.871           1.000


**Interpretación de la matriz de correlaciones:**
- **`views` ↔ `applies`:** correlación positiva esperada — más visibilidad genera más candidaturas.
- **`salary_annual` ↔ `employee_count`:** si positiva, las grandes empresas pagan más (economías de escala y presupuestos mayores).
- **`follower_count` ↔ `views`:** empresas con más seguidores en LinkedIn atraen más tráfico a sus ofertas.
- **`salary_annual` ↔ `views`:** correlación cercana a 0 indica que publicar el salario no determina significativamente la visibilidad de la oferta.

Nota: correlación no implica causalidad. Estos valores cuantifican relaciones lineales; relaciones no lineales pueden existir y no quedar capturadas aquí.

## 4. Análisis por grupos (.groupby() y pivot_table())

### 4.0 Índice de Gini

El Índice de Gini es una métrica que se utiliza para medir el nivel de desigualdad o de concentración de un recurso (como los salarios) dentro de una población determinada.

A diferencia de la desviación típica, que mide distancias absolutas, el Índice de Gini mide proporciones relativas.

El índice está acotado entre 0 y 1:

* Gini = 0 (Igualdad perfecta): Significa que el recurso está repartido de forma idéntica entre todos.

* Gini = 1 (Desigualdad perfecta): Significa que una sola persona concentra la totalidad del recurso, mientras que el resto no tiene nada.

Si se oredena los datos de manera ascendente, de modo que $x_1 \le x_2 \le \dots \le x_n$, la fórmula del Índice de Gini para una muestra es:

$$
G = \frac{\sum_{i=1}^{n} (2i - n - 1) \cdot x_i}{n \cdot \sum_{i=1}^{n} x_i}
$$

Donde:
- $n$: El tamaño de la muestra.
- $x_i$: El elemento en la posición $i$ de la muestra (una vez ordenados de menor a mayor).
- $\sum_{i=1}^{n} x_i$: La suma total de todos los elementos de la muestra.
- $(2i - n - 1)$: Es el factor de ponderación que da más peso a los elementos que se encuentran en las posiciones más altas de la muestra ordenada.

In [48]:
def gini_coef(series):
    """
    Calculates the Gini Coefficient for a single Pandas Series.
    Automatically drops missing values (NaN) and values <= 0.

    Parameters:
    series (pd.Series): The Pandas Series containing the numerical data.

    Returns:
    float: The Gini coefficient (between 0 and 1).
    """
    # Extract values, drop NaNs, and filter out non-positive values
    values = series.dropna().values
    values = values[values > 0]

    if len(values) == 0:
        raise ValueError("The series does not contain enough valid values greater than 0.")

    # Sort values in ascending order
    sorted_values = np.sort(values)
    n = len(sorted_values)

    # Calculate Gini coefficient
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * sorted_values) / (n * np.sum(sorted_values))) - ((n + 1) / n)

### 4.1 Salario por nivel de experiencia

In [49]:
if 'formatted_experience_level' in df_sal.columns:
    exp_sal = (df_sal.groupby('formatted_experience_level')['salary_annual']
               .agg(Media='mean',
                    Mediana='median',
                    N_ofertas='count',
                    Std='std',
                    Gini_Coef=gini_coef)
               .round({'Media': 0, 'Mediana': 0, 'Std': 0, 'Gini_Coef': 3}))

    seniority = ['executive', 'director', 'mid-senior level', 'associate',
                 'entry level','internship']
    exp_sal.index = pd.Categorical(exp_sal.index, categories=seniority, ordered=True)
    print("Salario anual por nivel de experiencia:")
    print(exp_sal.sort_index().to_string())

Salario anual por nivel de experiencia:
                     Media   Mediana  N_ofertas      Std  Gini_Coef
executive         278500.0  275000.0          3  20476.0      0.032
director          231156.0  225050.0         16  42761.0      0.101
mid-senior level  155535.0  145600.0        656  52236.0      0.189
associate         112087.0  110000.0         84  33719.0      0.166
entry level       137451.0  131600.0        114  46134.0      0.189
internship         81584.0   62400.0          9  64427.0      0.324


**Interpretación:** La escalera salarial (Entry < Mid-Senior < Director < Executive) es una de las variables más importantes para el diseño del programa de reskilling: cuantifica exactamente cuánto valor económico aporta cada nivel de cualificación. Si la diferencia entre Entry y Mid-Senior es grande, el ROI del reskilling es alto y justifica la inversión.

**Observaciones**:

1. **La "Trampa" del Associate vs. Entry Level:**
   Contrario a la intuición, el nivel `entry level` presenta una mediana salarial superior (131.600\$) a la del nivel `associate` (110.000\$). Esto sugiere una segmentación clara del mercado: mientras que el *entry level* técnico actúa como una puerta de entrada altamente competitiva (capturando perfiles de alto valor), el *associate* parece agrupar roles con menor exigencia técnica, esto último habría que analizarlo con más detalle porque no parece que tenga sentido.

2. **Polarización en niveles iniciales:**
   El grupo `entry level` presenta una desviación estándar alta (46.134\$) y un Gini significativo (0.189). Esto confirma que el mercado de primer empleo está polarizado: existen ofertas en empresas pequeñas con salarios básicos frente a posiciones en *Big Tech* que ofrecen salarios premium a recién graduados. Por el contrario, el grupo `associate` muestra una mayor estandarización (Gini 0.166), indicando que, tras los primeros años, las bandas salariales se vuelven más homogéneas y reguladas.

3. **Consistencia y predictibilidad en los puestos superiores (Directores y Ejecutivos):**
   A diferencia de lo que se podría esperar, los niveles más altos son los más estables. El nivel `executive` presenta el Gini más bajo de todo el dataset (0.032), lo que demuestra que, en esta muestra, los salarios ejecutivos son altamente predecibles y están sujetos a bandas salariales corporativas muy consolidadas. No se observa la volatilidad extrema que suele atribuirse al sector, sino una estructura de compensación técnica muy marcada.

4. **El salto estratégico:**
   Existe un diferencial significativo entre el nivel `mid-senior` (145.600\$) y `director` (225.050\$). Este salto de ~80.000\$ representa el mayor incentivo económico detectado, lo cual justifica el enfoque de los programas de reskilling hacia la especialización avanzada para escalar hacia posiciones de liderazgo técnico.

### 4.2 Número de ofertas y salario por tipo de contrato

In [50]:
wt_sal = (df_sal.groupby('formatted_work_type')['salary_annual']
           .agg(Media='mean',
                Mediana='median',
                N_ofertas='count',
                Std='std',
                Gini_Coef=gini_coef)
           .round({'Media': 0, 'Mediana': 0, 'Std': 0, 'Gini_Coef': 3})
           .sort_values('N_ofertas', ascending=False))
print("Por tipo de contrato:")
print(wt_sal.to_string())

Por tipo de contrato:
                        Media   Mediana  N_ofertas      Std  Gini_Coef
formatted_work_type                                                   
full-time            160364.0  151700.0        634  55186.0      0.195
contract             122840.0  124800.0        207  34680.0      0.160
other                152791.0  151075.0         11  38466.0      0.135
part-time            126045.0   83300.0         11  66462.0      0.274
temporary            156803.0  140400.0         10  41800.0      0.137
internship            72295.0   66295.0          9  30076.0      0.219


**Observaciones**:

El análisis de las modalidades de contratación revela cómo la naturaleza del vínculo laboral condiciona la equidad salarial y la estabilidad en el mercado de datos.

**1. La alta volatilidad y polarización del empleo a tiempo parcial (`part-time`):**
Este segmento destaca por ser el más desigual del dataset, con el coeficiente de Gini más elevado (0.274). Existe una divergencia significativa entre su media (126.045\$) y su mediana (83.300\$), lo cual confirma que el mercado *part-time* no es uniforme. La altísima desviación estándar (66.462\$) confirma que este grupo es muy heterogéneo, donde conviven posiciones de soporte con salarios bajos frente a consultorías especializadas de alto valor que, al ser pocas, distorsionan la media al alza. Es el segmento de mayor incertidumbre.

**2. Madurez y estandarización de los autónomos (`contract`):**
Contrario a la percepción común de inestabilidad, los datos muestran que el grupo de autónomos es notablemente sólido. Con un Gini de 0.160 —uno de los más bajos del análisis— y una proximidad muy estrecha entre su media (122.840\$) y su mediana (124.800\$), se confirma que estamos ante un mercado altamente profesionalizado y maduro. Las empresas han logrado estandarizar las tarifas por proyecto o servicio, convirtiendo esta modalidad en una opción predecible y equitativa para los profesionales de datos.

**3. El estándar corporativo del empleo a tiempo completo (`full-time`):**
Como columna vertebral del mercado, el segmento *full-time* presenta un coeficiente de Gini de 0.195, lo que sugiere una estructura salarial sana y regulada. La dispersión presente es una consecuencia natural de la jerarquía de experiencia (desde niveles *entry* hasta ejecutivos). Los datos confirman que el empleo indefinido mantiene una mayor consolidación en las bandas salariales corporativas, siendo el entorno más estable y predecible para el desarrollo de carrera en el sector.

**4. El segmento de prácticas (`internship`) como indicador de competitividad:**
Aunque presenta la mediana más baja (66.295\$), resulta llamativo que su coeficiente de Gini sea de 0.219, considerablemente superior al de los niveles de contratación autónoma. Esto revela que el mercado de *internships* no es una categoría de salario único, sino que existe una competencia real entre empresas: aquellas que ofrecen programas de prácticas de alto potencial con compensaciones competitivas para atraer talento joven, frente a programas tradicionales con remuneraciones menos atractivas.

### 4.3 Top industrias por ofertas y salario

In [51]:
if 'job_industries_list' in df_sal.columns:
    df_ind = df_sal[['job_id','salary_annual','job_industries_list']].dropna(subset=['job_industries_list']).copy()
    df_ind['industry'] = df_ind['job_industries_list'].str.split(', ')
    df_ind_exp = df_ind.explode('industry')

    ind_stats = (df_ind_exp.groupby('industry')['salary_annual']
                 .agg(Media='mean',
                      N_ofertas='count',
                      Std='std',
                      Gini_Coef=gini_coef)
                 .round({'Media': 0, 'Std': 0, 'Gini_Coef': 3})
                 .sort_values('N_ofertas', ascending=False)
                 .head(15))
    print("Top 15 industrias — roles de datos con salario:")
    print(ind_stats.to_string())

Top 15 industrias — roles de datos con salario:
                                     Media  N_ofertas      Std  Gini_Coef
industry                                                                 
IT Services and IT Consulting     153974.0        183  51350.0      0.186
Financial Services                163423.0        125  53963.0      0.185
Software Development              179794.0        107  55236.0      0.174
Staffing and Recruiting           141635.0         64  50110.0      0.191
Technology                        166002.0         56  54890.0      0.184
Hospitals and Health Care         137871.0         51  50971.0      0.200
Banking                           152921.0         51  42221.0      0.153
Insurance                         142870.0         39  44060.0      0.171
Information Services              157088.0         37  53501.0      0.189
Information and Media             145529.0         36  41189.0      0.157
Business Consulting and Services  148054.0         28  62101.0  

**Observaciones**:

El análisis de las 15 industrias principales confirma que, aunque el mercado de datos es globalmente robusto, existen divergencias notables en cómo cada sector valora y estructura la compensación de sus profesionales.

**1. Estandarización y madurez del mercado:**
La mayoría de las industrias presentan un coeficiente de Gini bastante bajo (en el rango de 0.150 a 0.190), lo que confirma que estamos ante mercados altamente estandarizados. Las bandas salariales para roles de datos están consolidadas; la progresión profesional es predecible y la remuneración es homogénea dentro de cada sector. Esta estabilidad sugiere que el precio de mercado por rol de datos es una variable bien definida para los departamentos de Recursos Humanos.

**2. Dominios de alto impacto y alta remuneración:**
El sector de *Information and Internet* destaca como el líder absoluto en competitividad, registrando la media salarial más alta (202.855\$) con un Gini muy controlado (0.158). Le siguen de cerca industrias como *Software Development* y *Telecommunications*, que actúan como motores de la demanda. Estos sectores no solo concentran gran parte de las ofertas, sino que han logrado optimizar sus escalas salariales para atraer talento especializado con una equidad interna superior al promedio.

**3. El ecosistema bancario y financiero:**
Industrias como *Banking* (Gini: 0.153) y *Financial Services* (Gini: 0.185) muestran una estructura de compensación muy eficiente. En particular, la banca presenta uno de los niveles de desigualdad más bajos de toda la muestra, lo que refleja políticas de contratación rígidas y bien reguladas, típicas de entornos con altos requerimientos de cumplimiento y seguridad.

**4. La polarización en sectores de nicho y tradicionales:**
A diferencia de los sectores tecnológicos, la industria de *Business Consulting and Services* presenta una mayor desigualdad interna (Gini: 0.210) y una dispersión salarial significativa. Esto puede deberse a la variabilidad de proyectos: la consultoría demanda perfiles tanto tácticos como estratégicos (desde analistas junior hasta arquitectos de datos de alto nivel), lo que ensancha la brecha salarial. Por otro lado, *Higher Education* se posiciona como un caso atípico con la media salarial más baja (89.404\$) y una dispersión muy reducida (Std: 25.419\$), lo cual es coherente con estructuras universitarias donde los salarios suelen estar vinculados a tablas retributivas rígidas.

**5. Consideraciones sobre Salud:**
Dentro de este ranking, *Hospitals and Health Care* mantiene una estructura salarial que sigue mostrando mayor desigualdad (Gini: 0.200) comparado con el sector financiero o bancario. Esto confirma que, en el ámbito clínico, los datos siguen siendo un sector polarizado: conviven funciones administrativas de gestión de registros con roles de alta especialización (IA médica o bioinformática), lo que genera una brecha salarial necesaria para retener al talento altamente cualificado frente a la oferta de las *Big Tech*.

### 4.4 Top áreas de negocio

In [52]:
if 'job_skills_list' in df.columns:
    df_sk = df[['job_id','job_skills_list']].dropna(subset=['job_skills_list']).copy()
    df_sk['skill'] = df_sk['job_skills_list'].str.split(', ')
    df_sk_exp = df_sk.explode('skill')
    skill_counts = df_sk_exp['skill'].value_counts().head(15)
    print("Top 15 áreas de negocio en roles de datos:")
    for sk, cnt in skill_counts.items():
        pct = cnt / len(df) * 100
        print(f"  {str(sk):<35} {cnt:>6,}  ({pct:.1f}%)")

Top 15 áreas de negocio en roles de datos:
  Information Technology               2,089  (71.6%)
  Engineering                            775  (26.6%)
  Analyst                                658  (22.6%)
  Research                               453  (15.5%)
  Business Development                   164  (5.6%)
  Sales                                  155  (5.3%)
  Other                                  118  (4.0%)
  Finance                                 80  (2.7%)
  Consulting                              70  (2.4%)
  Management                              48  (1.6%)
  Strategy/Planning                       40  (1.4%)
  Project Management                      39  (1.3%)
  Product Management                      32  (1.1%)
  Quality Assurance                       28  (1.0%)
  Manufacturing                           27  (0.9%)


**Observaciones**:

El análisis de las áreas de negocio donde se integran los roles de datos revela una realidad fundamental para el *reskilling*: **el talento en datos no es un ente aislado, sino un activo transversal** que debe asentarse sobre dominios de negocio específicos para maximizar su impacto.

**1. La hegemonía tecnológica (Infraestructura y Base Técnica):**
La gran mayoría de las oportunidades (casi el 72% en *Information Technology* y un 26% en *Engineering*) confirma que el destino natural de un perfil de datos sigue siendo la infraestructura técnica. Para un programa de *reskilling*, esto implica que los alumnos deben dominar el ciclo de vida del dato (arquitectura, calidad y despliegue) como requisito indispensable antes de avanzar hacia la analítica avanzada.

**2. El rol del analista como perfil híbrido:**
La alta demanda de perfiles de *Analyst* (22.6%) confirma la necesidad crítica de profesionales que actúen como puente. El mercado busca perfiles capaces de traducir métricas técnicas a KPIs estratégicos, lo que valida la importancia de integrar formación en *Data Storytelling* y visualización de datos dentro del programa.

**3. El valor añadido del dominio vertical:**
La distribución muestra que las habilidades puramente técnicas no son suficientes por sí solas. La presencia significativa de áreas como *Business Development*, *Sales*, *Finance* y *Strategy* indica que **el mayor valor añadido del profesional de datos moderno es el dominio del contexto de negocio**. Un alumno que combina conocimientos técnicos con un entendimiento profundo de *Finance* o *Business Strategy* es más valioso y difícil de reemplazar que un perfil puramente técnico.

### 4.5 Competencias técnicas más demandas

In [53]:
# Convert the string representation back to an actual Python list object
df_sal['hard_skills'] = df_sal['hard_skills'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else []
)

exploded_df_sal = df_sal.explode('hard_skills')

roles = ['data engineer', 'data scientist', 'data analyst']
all_skills_list = []

for role in roles:
    # Filter data for the current role
    role_df = exploded_df_sal.query("norm_role == @role")
    total_de_jobs = role_df.index.nunique()

    # Process the data
    skills_df = (
        role_df
        .groupby(['norm_role', 'hard_skills'])
        .agg(
            count=('hard_skills', 'size'),
            percentage=('hard_skills', lambda x: (x.size / total_de_jobs) * 100),
            median=('salary_annual', 'median'),
            iqr=('salary_annual', lambda x: np.percentile(x.dropna(), 75) - np.percentile(x.dropna(), 25))
        )
        .round({'percentage': 2, 'median': 0, 'iqr': 0})
        .sort_values(by='count', ascending=False)
        .reset_index()
    )

    # Append the processed dataframe to our list
    all_skills_list.append(skills_df)

    # Print title and table
    print(f"{role.upper()}")
    print()
    print(skills_df.drop(columns=['norm_role']).head(10))
    print("-" * 55 + "\n")

DATA ENGINEER

  hard_skills  count  percentage    median      iqr
0      python    175       58.33  174000.0  62000.0
1         sql    160       53.33  158550.0  53825.0
2         aws    102       34.00  175900.0  61450.0
3       azure     85       28.33  173675.0  47200.0
4       spark     77       25.67  176800.0  69600.0
5        java     60       20.00  180000.0  60900.0
6       scala     52       17.33  176800.0  59700.0
7         api     50       16.67  168200.0  50850.0
8         gcp     48       16.00  176800.0  59548.0
9   snowflake     46       15.33  176800.0  37670.0
-------------------------------------------------------

DATA SCIENTIST

     hard_skills  count  percentage    median      iqr
0         python     91       57.59  162600.0  90250.0
1     statistics     89       56.33  191900.0  91700.0
2            sql     61       38.61  162600.0  77600.0
3              r     61       38.61  189200.0  91400.0
4            nlp     29       18.35  162600.0  73000.0
5         

**Observaciones**:

El análisis de las competencias técnicas revela tres perfiles claramente diferenciados, donde la combinación de herramientas define no solo la responsabilidad del puesto, sino también la volatilidad y estructura de la compensación salarial.

1. **Data Engineer**: La infraestructura como base.
El perfil de *Data Engineer* es el más estandarizado y centrado en la arquitectura cloud.
* **Dominio técnico:** Python (58%) y SQL (53%) son la base operativa. La demanda de ecosistemas de datos a gran escala como AWS (34%), Azure (28%) y Spark (25%) muestra una especialización técnica sólida y constante.
* **Predictibilidad salarial:** La baja dispersión (IQR) en herramientas como Snowflake (37.6k$) indica que el mercado tiene muy claro cuánto pagar por estas habilidades, configurando un entorno de remuneración estable y predecible.

2. **Data Scientist**: Especialización científica y alta volatilidad.
El *Data Scientist* se aleja del software tradicional para enfocarse en la investigación y el modelado, lo que conlleva una mayor desigualdad salarial.
* **El ecosistema científico:** La combinación de Python (57%) y Estadística (56%) es el núcleo. Es llamativa la alta demanda de R (38%) y NLP (18%), que denotan una orientación clara hacia el modelado avanzado y la inteligencia artificial.
* **Dispersión salarial:** Este rol presenta los **IQR más altos de toda la muestra** (superando los 90k\$ en estadística y R). Esto sugiere que el salario depende menos de la herramienta específica y más de la empresa (Big Tech vs. industria tradicional) y del nivel de especialización en áreas críticas como *Deep Learning*.

3. **Data Analyst**: El nexo con el negocio
El *Data Analyst* actúa como el perfil más equilibrado, donde las herramientas de visualización tienen el mismo peso que el análisis técnico.
* **Dominio de la visualización:** La alta demanda de SQL (44%) combinada con herramientas de BI como Tableau (23%) y Power BI (21%) confirma que el analista es el principal traductor de datos a insights estratégicos.
* **Equidad salarial:** Este perfil muestra una estructura salarial mucho más compacta (IQR rondando los 45k\$-50k\$). La estandarización de estas herramientas reduce la incertidumbre salarial, creando bandas retributivas más estrechas y coherentes que en la ciencia de datos.

En conclusión, para maximizar la empleabilidad, el programa de reskilling debe diferenciarse según el objetivo del alumno:

* **Para perfiles de ingeniería:** La prioridad debe ser el ecosistema Cloud (AWS/Azure/GCP) y Spark; es donde reside el mayor volumen de ofertas y la demanda más estable.
* **Para perfiles analíticos:** El dominio de SQL junto con una herramienta de BI (Tableau o Power BI) es la ruta más directa a la inserción laboral.
* **Para perfiles científicos:** El éxito reside en la especialización (NLP, Deep Learning). Dado que el mercado es más volátil, el valor del profesional aquí depende de la capacidad de aplicar técnicas avanzadas que justifiquen las altas bandas salariales.

### 4.6 Tabla pivot: mediana salarial por experiencia × tipo de contrato

In [54]:
# Define seniority and contract-types order
seniority_order = ['executive', 'director', 'mid-senior level', 'associate', 'entry level', 'internship']
contract_types_order = ['full-time', 'contract', 'part-time', 'temporary', 'internship', 'other']

try:
    pivot = pd.pivot_table(
        df_sal,
        values='salary_annual',
        index='formatted_experience_level',
        columns='formatted_work_type',
        aggfunc='median',
        fill_value=0
    ).round(0)
    # Reindex both axes
    ordered_pivot = pivot.reindex(index=seniority_order, columns=contract_types_order, fill_value=0)
    print("Tabla pivot — Mediana salarial (USD) por experiencia y tipo de contrato:")
    print(ordered_pivot.to_string())
except Exception as e:
    print(f"Nota: {e}")

Tabla pivot — Mediana salarial (USD) por experiencia y tipo de contrato:
formatted_work_type         full-time  contract  part-time  temporary  internship     other
formatted_experience_level                                                                 
executive                    275000.0       0.0        0.0        0.0         0.0       0.0
director                     225050.0       0.0        0.0        0.0         0.0       0.0
mid-senior level             159775.0  133120.0    83300.0   166379.0    100000.0  104000.0
associate                    112000.0  100714.0    70720.0   130000.0         0.0       0.0
entry level                  137800.0   97760.0   156860.0        0.0    124800.0  162375.0
internship                   154200.0       0.0        0.0        0.0     49920.0       0.0


## 5. Probabilidad condicional P(A|B)

In [55]:
# P(nivel senior | rol Data Scientist) vs P(senior | Data Engineer) vs P(senior | global)
df_p = df.copy()
df_p['is_senior'] = df_p['formatted_experience_level'].str.contains(
    'senior|director|executive', case=False, na=False
)
df_p['is_ds'] = df_p['title'].str.contains('scientist|science', case=False, na=False)
df_p['is_de'] = df_p['title'].str.contains('data engineer', case=False, na=False)
df_p['is_da'] = df_p['title'].str.contains('data analyst', case=False, na=False)

p_s_ds  = df_p.loc[df_p['is_ds'], 'is_senior'].mean()
p_s_de  = df_p.loc[df_p['is_de'], 'is_senior'].mean()
p_s_da  = df_p.loc[df_p['is_da'], 'is_senior'].mean()
p_s_all = df_p['is_senior'].mean()

print("=== Probabilidad Condicional — P(Senior | Rol) ===")
print(f"  P(Senior | Data Scientist) = {p_s_ds:.3f}  ({p_s_ds*100:.1f}%)")
print(f"  P(Senior | Data Engineer)  = {p_s_de:.3f}  ({p_s_de*100:.1f}%)")
print(f"  P(Senior | Data Analyst)   = {p_s_da:.3f}  ({p_s_da*100:.1f}%)")
print(f"  P(Senior | cualquier rol)  = {p_s_all:.3f}  ({p_s_all*100:.1f}%)")

=== Probabilidad Condicional — P(Senior | Rol) ===
  P(Senior | Data Scientist) = 0.755  (75.5%)
  P(Senior | Data Engineer)  = 0.819  (81.9%)
  P(Senior | Data Analyst)   = 0.615  (61.5%)
  P(Senior | cualquier rol)  = 0.797  (79.7%)


**Interpretación:** La probabilidad condicional P(Senior | Data Scientist) nos dice qué porcentaje de las ofertas de Data Scientist requieren nivel senior. Comparada con la probabilidad marginal P(Senior), revela si el rol demanda más experiencia que la media. Esto es información concreta para que DataTalent diseñe rutas de reskilling realistas: no tiene sentido orientar candidatos junior a posiciones que estadísticamente exigen senior.

## 6. Detección de Sesgos en el Dataset

Los sesgos en los datos pueden llevar a decisiones empresariales erróneas y a modelos de IA discriminatorios. Identificamos **4 sesgos** con su origen y su impacto potencial.

### Sesgo 1: MNAR en datos salariales (*Missing Not At Random*)
**Descripción:** Las columnas salariales tienen entre el 71% y el 95% de valores nulos. Los datos no faltan aleatoriamente: existe una razón sistemática para que falten.

**Origen:** Las empresas eligen estratégicamente si publicar el salario. Las startups y PYMEs con salarios menos competitivos suelen ocultarlos para no desalentar candidatos.


In [56]:
total = len(df)
con_salario = df['salary_annual'].notna().sum()
sin_salario = total - con_salario

print("=== Análisis MNAR — Datos Salariales ===")
print(f"Con salario publicado:    {con_salario:>7,}  ({con_salario/total*100:.1f}%)")
print(f"Sin salario publicado:    {sin_salario:>7,}  ({sin_salario/total*100:.1f}%)")
print()
# Do larger companies publish salaries more often?
if 'comp_size' in df.columns:
    print("Tasa de publicación de salario por tamaño de empresa:")
    res = df.groupby('comp_size')['salary_annual'].apply(
        lambda x: f"{x.notna().mean()*100:.0f}%  ({x.notna().sum():,} de {len(x):,})"
    )
    print(res.to_string())

=== Análisis MNAR — Datos Salariales ===
Con salario publicado:        916  (31.4%)
Sin salario publicado:      2,000  (68.6%)

Tasa de publicación de salario por tamaño de empresa:
comp_size
1.0     29%  (73 de 248)
2.0     22%  (88 de 405)
3.0    34%  (102 de 302)
4.0     31%  (77 de 250)
5.0    34%  (174 de 508)
6.0     31%  (67 de 214)
7.0    35%  (295 de 849)


**Impacto en un modelo predictivo:** Un modelo entrenado solo con los datos con salario aprendería los salarios de empresas que voluntariamente los publican (generalmente las más grandes y con mejor remuneración). Consecuencia: **sobreestimaría sistemáticamente** los salarios del mercado real. DataTalent podría orientar a candidatos con expectativas infladas, generando frustración en el proceso de selección.

### Sesgo 2: Sesgo geográfico — El dataset no representa el mercado español
**Descripción:** El dataset proviene de LinkedIn global, con predominancia de ofertas en EEUU. Si se usa para orientar candidatos en España, los hallazgos salariales y de skills no son directamente aplicables.

**Origen:** El scraping de LinkedIn se realizó principalmente en mercados anglosajones (mayor penetración de LinkedIn). Las ofertas españolas están subrepresentadas.

In [57]:
if 'comp_country' in df.columns:
    country_counts = df['comp_country'].value_counts().head(12)
    total_con_pais = df['comp_country'].notna().sum()
    print(f"Distribución geográfica (de {total_con_pais:,} registros con país):\n")
    for country, cnt in country_counts.items():
        pct = cnt / total_con_pais * 100
        bar = '█' * int(pct / 2)
        print(f"  {str(country):<30} {cnt:>6,}  ({pct:5.1f}%)  {bar}")
    print(f"\nRegistros sin país: {df['comp_country'].isna().sum():,}")

Distribución geográfica (de 2,892 registros con país):

  us                              2,577  ( 89.1%)  ████████████████████████████████████████████
  gb                                 82  (  2.8%)  █
  in                                 71  (  2.5%)  █
  0                                  61  (  2.1%)  █
  ca                                 41  (  1.4%)  
  de                                 10  (  0.3%)  
  oo                                  6  (  0.2%)  
  fr                                  6  (  0.2%)  
  dk                                  6  (  0.2%)  
  ae                                  6  (  0.2%)  
  sg                                  4  (  0.1%)  
  nl                                  4  (  0.1%)  

Registros sin país: 24


**Impacto en un modelo predictivo:** Un modelo entrenado con este dataset y aplicado al mercado español **transladaría estándares salariales estadounidenses** (generalmente mucho más altos) a un contexto europeo, generando expectativas irreales. Las skills demandadas también pueden diferir: el mercado español tiene mayor presencia de banca, turismo y sector público.

**Medida de mitigación:** Complementar con datos de Infojobs, Tecnoempleo o SEPE para el contexto español.

### Sesgo 3: Sesgo de selección — Solo ofertas publicadas en LinkedIn
**Descripción:** LinkedIn es dominante para perfiles tech, pero no representa el mercado completo. Infojobs, Indeed, portales gubernamentales y ofertas directas (sin plataforma) quedan excluidos.

**Origen:** El dataset se extrajo de una única fuente, creando un sesgo estructural hacia empresas con marca empleadora activa en LinkedIn.

In [58]:
print("=== Sesgo de Selección — Fuente única ===")
print("Plataforma: LinkedIn Job Postings (Kaggle)")
print()
print("Empresas sobrerepresentadas: grandes tecnológicas con marca empleadora fuerte")
print("Empresas subrepresentadas:  PYMEs, sector público, consultoras pequeñas")
print()
if 'comp_size' in df.columns:
    print("Distribución de ofertas por tamaño de empresa:")
    cs = df['comp_size'].value_counts(dropna=False)
    for size, cnt in cs.items():
        pct = cnt / len(df) * 100
        print(f"  {str(size):<15} {cnt:>7,}  ({pct:.1f}%)")

=== Sesgo de Selección — Fuente única ===
Plataforma: LinkedIn Job Postings (Kaggle)

Empresas sobrerepresentadas: grandes tecnológicas con marca empleadora fuerte
Empresas subrepresentadas:  PYMEs, sector público, consultoras pequeñas

Distribución de ofertas por tamaño de empresa:
  7.0                 849  (29.1%)
  5.0                 508  (17.4%)
  2.0                 405  (13.9%)
  3.0                 302  (10.4%)
  4.0                 250  (8.6%)
  1.0                 248  (8.5%)
  6.0                 214  (7.3%)
  nan                 140  (4.8%)


**Impacto en un modelo predictivo:** El modelo aprendería que "una oferta de datos típica proviene de una gran empresa tech con perfil LinkedIn activo". Fallaría en predecir salarios y skills para roles en PYMEs o sector público, que representan una parte significativa del tejido empresarial español (especialmente relevante para candidatos de ciudades no metropolitanas).

### Sesgo 4: Ausencia de datos de género (*Undisclosed protected attribute*)
**Descripción:** El dataset no contiene información de género. Sin este dato, es imposible detectar directamente brechas salariales de género, uno de los análisis más críticos para garantizar equidad en el reskilling.

**Origen:** LinkedIn no incluye género por defecto en sus datos de ofertas de empleo. Incorporarlo requeriría autodeclaración o inferencia algorítmica, ambas con problemas legales y éticos bajo el RGPD europeo.

In [59]:
print("=== Sesgo por Atributo Protegido Ausente (Género) ===")
print()
print("Variables de género: NO disponibles en el dataset")
print()
print("Consecuencias directas:")
print("  1. No podemos cuantificar la brecha salarial de género")
print("  2. Un modelo entrenado sin control de género puede perpetuar sesgos históricos")
print("  3. Las decisiones de reskilling sin perspectiva de género pueden ser excluyentes")
print()
# Proxy: distribution of roles that have historically shown gender bias
if 'title' in df.columns:
    tech_roles = df['title'].str.contains('engineer|developer|architect', case=False, na=False).sum()
    soft_roles = df['title'].str.contains('analyst|coordinator|manager', case=False, na=False).sum()
    print(f"Roles de ingeniería/desarrollo (históricamente masculinizados): {tech_roles:,} ({tech_roles/len(df)*100:.1f}%)")
    print(f"Roles analíticos/gestión (mayor diversidad de género): {soft_roles:,}  ({soft_roles/len(df)*100:.1f}%)")
    print()
    print("Nota: esta es una aproximación indirecta, no un análisis de género real.")

=== Sesgo por Atributo Protegido Ausente (Género) ===

Variables de género: NO disponibles en el dataset

Consecuencias directas:
  1. No podemos cuantificar la brecha salarial de género
  2. Un modelo entrenado sin control de género puede perpetuar sesgos históricos
  3. Las decisiones de reskilling sin perspectiva de género pueden ser excluyentes

Roles de ingeniería/desarrollo (históricamente masculinizados): 1,097 (37.6%)
Roles analíticos/gestión (mayor diversidad de género): 1,384  (47.5%)

Nota: esta es una aproximación indirecta, no un análisis de género real.


**Impacto en un modelo predictivo:** Un modelo de predicción salarial sin control de género puede aprender patrones discriminatorios implícitos: si históricamente las mujeres están subrepresentadas en roles senior de ingeniería de datos, el modelo aprenderá a predecir salarios más bajos para ciertos patrones de carrera. Esto infringiría el principio de equidad algorítmica (Fairness AI) y podría vulnerar legislación anti-discriminación europea.

**Medida de mitigación:** Cruzar con encuestas que incluyan autodeclaración de género (Stack Overflow Developer Survey, Kaggle ML Survey) para análisis de equidad de género.

## 7. Resumen del análisis estadístico

| Hallazgo | Métrica clave | Implicación para DataTalent |
|----------|-------------|----------------------------|
| Salario mediano en roles de datos | Ver análisis § 2.1 | Referencia para orientación salarial realista |
| Skills más demandadas | Top 5 del ranking | Define el currículo mínimo del programa de reskilling |
| ROI del nivel de experiencia | Diferencia median Entry vs Senior | Argumento económico para justificar la inversión en formación |
| Industrias con más demanda | Top 3 de § 4.3 | Sectores objetivo para colocación de candidatos |
| **MNAR salarial** | 71–95% de nulos | Las cifras salariales son una estimación parcial sesgada hacia arriba |
| **Sesgo geográfico** | Mayoría EEUU | Ajustar o complementar con datos españoles antes de usar |
| **Sin datos de género** | 0% cobertura | Requiere fuente externa para garantizar equidad en el programa |

**Próximo paso (Fase 4):** Visualización de todos estos hallazgos.